In [ ]:
# 0. 패키지 버전 맞추기 — Colab에서 처음 한 번: 이 셀 실행 → 런타임 다시 시작 → 다음 셀부터
# (google-colab·numba·cudf 버전 경고와 bottleneck 경고는 무시해도 된다)
# 로컬은 .venv에 requirements.txt로 이미 설치되어 있으면 건너뛴다: python -m pip install -r requirements.txt
import sys
if 'google.colab' in sys.modules:
    !pip install -q pandas==3.1.0 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 xgboost==3.4.1 lightgbm==4.7.0 pyarrow==26.0.0 PyYAML==6.0.3
    print('설치 끝: 런타임 → 세션 다시 시작 후 다음 셀부터 실행')
else:
    print('로컬: 설치 생략 (requirements.txt 버전 사용)')

# model_test — linear_model (OLS·Ridge)

v2 모델 비교의 선형 부분. 설계는 `config/experiment.yaml`의 `MODEL_GRID`와 노션 「11주차 v2 모델」. 계산 코드는 `src/evaluation/model_test.py`·`src/models/linear.py`에 있고 이 노트북은 함수만 부른다.

- 입력: `data/interim`(panel, eligibility)·`data/processed`(h1_components, h3)와 config만. 이전 실행 결과를 읽지 않는다.
- **이어서 돌리기 없음**: 결과는 이 세션 메모리에 쌓이고 마지막 셀에서 한 번에 검사·저장한다. 커널이 끊기면 위에서부터 다시 실행한다.
- 셀 순서: 0 환경 확인 → 1 경로·설정 → 2 입력 준비 → 3~8 모델 × 손실 6개 → 9 합치기·저장.
- 로컬(.venv 커널, CPU 병렬)과 Colab 모두에서 돌 수 있게 1번 셀이 경로를 나눈다.

## 0. 실행 환경 확인

계산은 하지 않고 위치·파일·GPU·버전만 확인한다.

In [ ]:
import os, sys, platform, subprocess, importlib.util, shutil
from pathlib import Path

print('python      :', sys.version.split()[0], platform.platform())
print('작업 폴더    :', os.getcwd())
print('/content 존재:', Path('/content').exists())

# 저장소가 보이는지: 위로 올라가며 src/와 config/experiment.yaml을 찾는다
found = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file()), None)
print('저장소 루트   :', found)

if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], capture_output=True, text=True).stdout)
else:
    print('GPU          : 없음(선형 모델은 CPU만 사용)')

for m in ['numpy', 'pandas', 'pyarrow', 'scipy', 'sklearn', 'xgboost', 'lightgbm', 'yaml']:
    if importlib.util.find_spec(m):
        mod = __import__(m)
        print(f'{m:9s}', getattr(mod, '__version__', '?'))
    else:
        print(f'{m:9s}', '설치 안 됨')

In [ ]:
import os, subprocess
print('CPU 개수:', os.cpu_count())
if os.name != 'nt':   # Linux(Colab)에서만
    print(subprocess.run(['nproc'], capture_output=True, text=True).stdout.strip())
    print(subprocess.run(['free', '-g'], capture_output=True, text=True).stdout)

## 1. 경로·설정

- **로컬**: 이 노트북이 있는 저장소를 루트로 쓴다. 결과는 `outputs/model_results/<작성자>/<시각>_base_linear/`(팀 공유 규격).
- **Colab**: `GIT_REF`에 **브랜치 이름**(예: `main`, `hong`)을 적는다. 매번 GitHub에서 그 브랜치의 최신 코드를 받으므로, 로컬에서 고친 것은 **푸시한 뒤** 돌린다. 같은 결과를 다시 만들 때만 커밋 해시를 넣는다. `data/`는 내 드라이브의 `금융데이터인텔리전스`(공유 폴더 바로가기)에서 연결, 결과는 내 드라이브 `model_results/<AUTHOR>/`.
- `WORKERS`: Fold를 나눠 계산할 CPU 수. 로컬 8개 중 6개(나머지는 컴퓨터 사용용). 결과는 CPU 수와 무관하게 같다.

In [ ]:
import os, sys, re, subprocess
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JiunShin1214/finance_analytics.git'
GIT_REF = ''              # Colab에서만: 브랜치 이름(예: 'main', 'hong') — 그 브랜치의 GitHub 최신 코드. 특정 시점 재현은 커밋 해시
AUTHOR = ''              # 결과 폴더 outputs/model_results/<작성자>/ (영문·숫자·_·-만)
INPUT_TRANSFORM = 'zscore'   # 선형 입력: 'zscore'(매달 z점수, 2026-11-16 튜닝 7 결정) | 'rank'(매달 순위 [−1,1], 이전 기본)
if not AUTHOR:
    raise ValueError('AUTHOR를 채운다(결과 폴더 outputs/model_results/<AUTHOR>/).')
DRIVE_BASE = Path('/content/drive/MyDrive/금융데이터인텔리전스')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not GIT_REF:
        raise RuntimeError("GIT_REF에 브랜치 이름(예: 'main', 'hong')을 적는다.")
    ROOT = Path('/content/finance_analytics')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    # GitHub에서 최신을 받아 GIT_REF로 맞춘다: 브랜치 이름이면 원격 최신(origin/<브랜치>), 아니면 커밋 해시
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', '--prune'], check=True)
    _remote = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--verify', '--quiet', f'origin/{GIT_REF}'],
                             capture_output=True, text=True).stdout.strip()
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', _remote or GIT_REF], check=True)
    print('받은 코드:', GIT_REF, '@', subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--short', 'HEAD'],
                                               capture_output=True, text=True, check=True).stdout.strip())
    if not (ROOT / 'data').is_symlink():       # 저장소의 data/(.gitkeep만 있음)를 Drive 공유 data로 바꿔 연결
        (ROOT / 'data').rename(ROOT / 'data_repo_placeholder')
        (ROOT / 'data').symlink_to(DRIVE_BASE / 'data')
    OUT_ROOT = Path('/content/drive/MyDrive/model_results')   # Colab: 내 드라이브(개인)에 같은 구조로 → 내려받아 로컬 outputs/model_results/에 넣고 커밋
    WORKERS = max(1, (os.cpu_count() or 2))
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
    OUT_ROOT = None    # 기본: ROOT/outputs/model_results (<작성자>/<결과 폴더>)
    WORKERS = 6

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
import pandas as pd
from src.evaluation import model_test as mt

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))

# requirements.txt 고정 버전과 설치 버전 비교(다르면 경고)
pins = dict(re.findall(r'^([A-Za-z0-9_.-]+)==([^\s#]+)', (ROOT / 'requirements.txt').read_text(encoding='utf-8'), re.M))
for name in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'pyarrow', 'PyYAML']:
    try:
        got = version(name)
    except PackageNotFoundError:
        got = '없음'
    print(f"{name:13s} 고정 {pins.get(name, '?'):8s} 설치 {got:8s} {'' if got == pins.get(name) else '← 다름'}")

started = datetime.now()
pd.set_option('display.width', 200, 'display.max_rows', 200, 'display.float_format', '{:.4f}'.format)
print('ROOT   :', ROOT)
print('저장   :', (OUT_ROOT or ROOT / config['PATHS']['model_results']) / AUTHOR)
print('WORKERS:', WORKERS, '| 시작:', started.isoformat(timespec='seconds'))

## 2. 입력 준비 (약 2분)

λ 후보 10개의 σ를 수익률로 다시 계산하고, 매달 순위 [−1, 1] 입력과 목표값 3종(그 달 학습 행 기준)을 만든다. 확인할 값: 검증 예측 133,876행, 학습 시작 2001년(`train_start_target_year`)과 학습 행 수.

In [ ]:
import time
t0 = time.perf_counter()
prepared = mt.prepare_inputs(ROOT, config, input_transform=INPUT_TRANSFORM)
print(f'입력 준비 {time.perf_counter() - t0:.0f}초')
for k, v in prepared['checks'].items():
    print(f'  {k:45s} {v}')
print('λ 후보:')
print(prepared['lambda_candidates'].round(4).to_string(index=False))
results = {}   # 셀(모델 × 손실)별 결과를 이 세션 메모리에 모은다

## 3~8. 모델 × 손실 (셀마다 따로 계산)

셀 하나 = 모델 하나 × 손실 하나 × 목표값 3종(원값·윈저·월별 표준화) × 입력 조합 15개(개별형, 지수형은 제외) × 16 Fold. 조정값(H3 λ, Ridge λ)은 Fold마다 내부 검증(공통 표본) 평균 IC 최고 한 쌍으로 고르고, 고른 뒤 다시 학습하지 않는다.

- 진행 표시: Fold 안 입력 조합마다 한 줄(경과·남은 예상), Fold가 끝나면 한 줄. 병렬이라 Fold 순서가 섞여 나온다.
- 끝나면 요약 표: 입력 조합 × 목표값별 바깥 검증 평균 IC(전체 종목·공통 표본), Fold 평균 학습 IC, 차이. 진행 확인용이며 최종 표는 9번 셀이 만든다.
- 예상 시간은 2026-11-13 로컬(CPU 6개) 실측 기준. 같은 셀을 다시 실행하면 그 셀 결과만 새로 계산해 덮어쓴다.

In [ ]:
ESTIMATE_MIN = {('ols', 'mse'): 0.4, ('ols', 'huber_q999'): 0.7, ('ols', 'huber_sigma135'): 2.5,
                ('ridge', 'mse'): 0.8, ('ridge', 'huber_q999'): 1.5, ('ridge', 'huber_sigma135'): 3.2}


def run_part(model, loss):
    """셀 하나(모델 × 손실)를 계산해 results에 넣고 요약 표를 보여 준다."""
    print(f'[{model} × {loss}] 예상 약 {ESTIMATE_MIN[(model, loss)]}분 (로컬 CPU 6개 실측 기준), 시작 {datetime.now():%H:%M:%S}')
    t = time.perf_counter()
    results[(model, loss)] = mt.run_linear(prepared, config, settings=mt.part_settings(model, loss), workers=WORKERS)
    print(f'[{model} × {loss}] 완료 {(time.perf_counter() - t) / 60:.1f}분')
    display(mt.part_summary(results[(model, loss)]))
    sel = results[(model, loss)]['selections']
    cols = ['h3_span', 'ridge_lambda', 'ties', 'huber_sigma_converged']
    print('Fold별 선택값 분포:', {c: sel[c].value_counts(dropna=False).to_dict() for c in cols if sel[c].notna().any()})

In [ ]:
# 3. OLS × MSE (예상 0.4분)
run_part('ols', 'mse')

In [ ]:
# 4. OLS × Huber① (δ = |목표값 − 그 달 평균|의 99.9% 분위수, 예상 0.7분)
run_part('ols', 'huber_q999')

In [ ]:
# 5. OLS × Huber② (δ = 1.35 × HuberRegressor σ, 예상 2.5분)
run_part('ols', 'huber_sigma135')

In [ ]:
# 6. Ridge × MSE (Ridge λ 6개 × H3 λ 10개 함께 고르기, 예상 0.8분)
run_part('ridge', 'mse')

In [ ]:
# 7. Ridge × Huber① (예상 1.5분)
run_part('ridge', 'huber_q999')

In [ ]:
# 8. Ridge × Huber② (예상 3.2분)
run_part('ridge', 'huber_sigma135')

## 9. 합치기 → 저장 (약 1분)

6개 셀 결과를 메모리에서 합쳐 표·예측값을 저장한다(검사 없음). IC 개월 수(`months`)·채점 행 수(`scored_rows`)는 표에 숫자로 나온다.

In [ ]:
missing = [p for p in mt.LINEAR_PARTS if p not in results]
if missing:
    raise RuntimeError(f'아직 안 돌린 셀: {missing}')
merged = mt.merge_results([results[p] for p in mt.LINEAR_PARTS])
final = mt.finalize_linear(ROOT, config, prepared, merged, started=started, out_root=OUT_ROOT, author=AUTHOR, experiment=f'base_{INPUT_TRANSFORM}')
print('저장:', final['run_dir'])
perf = final['tables']['performance_summary']
show = ['model', 'target', 'loss', 'input_set', 'mean_ic', 'icir', 'ic_positive_share', 'train_ic_fold_mean', 'train_minus_valid', 'r2_pct']
print('바깥 검증 평균 IC 상위 15개 (전체 종목):')
display(perf[perf.scope.eq('full')].sort_values('mean_ic', ascending=False)[show].head(15))
print('입력 H1+H3(개별형) 18개 설정 (전체 종목):')
display(perf[perf.scope.eq('full') & perf.input_set.eq('ind:size+bm+mom+sigma')][show])